In [ ]:
import os, sys, json, time
import numpy as np
import warnings
warnings.filterwarnings("ignore")

PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.config import DATASET_ORDER, N_FOLDS_JUSTIFIKASI, N_REPEATS_JUSTIFIKASI, N_FOLDS_FINAL, N_REPEATS_FINAL, SEED, TAB_DIR
from src.hdmc_multiview import (load_dataset, run_hdmc_pipeline, justifikasi_treatment_pair,
                                justifikasi_pemasangan_treatment,
                                HDMC_BACKBONE, HDMC_TREATMENTS, HDMC_META_LEARNER)
from src.justification_runner import (run_justification_pipeline, sintesis_pemilihan_treatment,
                                       sintesis_backbone_lintas_dataset)

print("Urutan dataset:", DATASET_ORDER)
print(f"Protokol justifikasi: n_folds={N_FOLDS_JUSTIFIKASI}, n_repeats={N_REPEATS_JUSTIFIKASI}")
print(f"Protokol final: n_folds={N_FOLDS_FINAL}, n_repeats={N_REPEATS_FINAL}")
print(f"HDMC terkunci: backbone={HDMC_BACKBONE}, treatments={HDMC_TREATMENTS}, meta={HDMC_META_LEARNER}")

In [ ]:
### Redirect output ke results_clean/ (tidak menimpa results/ asli)
import os
RESULTS_CLEAN_ROOT = os.path.abspath(os.path.join("..", "results_clean"))
for _sub in ["tables", "figures", "models", "predictions"]:
    os.makedirs(os.path.join(RESULTS_CLEAN_ROOT, _sub), exist_ok=True)

_new_tab_dir = os.path.join(RESULTS_CLEAN_ROOT, "tables")
_new_mdl_dir = os.path.join(RESULTS_CLEAN_ROOT, "models")
_new_pred_dir = os.path.join(RESULTS_CLEAN_ROOT, "predictions")
_new_fig_dir = os.path.join(RESULTS_CLEAN_ROOT, "figures")

import src.hdmc_multiview as _m1
import src.comparator_runner as _m2
import src.experiment_runner as _m3
import src.justification_runner as _m4
import src.data_quality as _m5

for _mod in [_m1, _m2, _m3, _m4, _m5]:
    if hasattr(_mod, "TAB_DIR"):
        _mod.TAB_DIR = _new_tab_dir
    if hasattr(_mod, "MDL_DIR"):
        _mod.MDL_DIR = _new_mdl_dir
    if hasattr(_mod, "PRED_DIR"):
        _mod.PRED_DIR = _new_pred_dir

TAB_DIR = _new_tab_dir
MDL_DIR = _new_mdl_dir
PRED_DIR = _new_pred_dir
FIG_DIR = _new_fig_dir

print(f"Output diarahkan ke: {RESULTS_CLEAN_ROOT}")

---
# BAGIAN B — EKSPERIMEN UTAMA HDMC

Protokol final (10-fold/1-repeat), IDENTIK dengan protokol yang akan dipakai di
nb03 (baseline+eksternal), supaya uji Wilcoxon berpasangan valid. Backbone,
treatment, dan meta-learner TERKUNCI (bukan hasil argmax dari Bagian A, lihat
instruksi proyek soal kriteria konsistensi lintas dataset).

Model backbone (2 treatment x 10 fold), model meta-learner (10 fold), OOF proba,
dan metrik lengkap SEMUA tersimpan ke results_clean/models/ dan results_clean/predictions/.

In [ ]:
hdmc_results = {}
t0_total = time.perf_counter()
for ds in DATASET_ORDER:
    t0 = time.perf_counter()
    agg = run_hdmc_pipeline(ds, verbose=True)
    elapsed = time.perf_counter() - t0
    hdmc_results[ds] = agg
    print(f"\n[{ds}] SELESAI dalam {elapsed/60:.1f} menit. "
          f"macro_f1_mean={agg['macro_f1_mean']:.4f} (std={agg['macro_f1_std']:.4f})")

total_elapsed = time.perf_counter() - t0_total
print(f"\n{'='*70}\nSELURUH PIPELINE HDMC SELESAI dalam {total_elapsed/60:.1f} menit "
      f"({total_elapsed/3600:.2f} jam)\n{'='*70}")

### Ringkasan hasil HDMC, 4 dataset

In [ ]:
import pandas as pd
rows = []
for ds, agg in hdmc_results.items():
    rows.append({
        "dataset": ds,
        "macro_f1_mean": round(agg["macro_f1_mean"], 4),
        "macro_f1_std": round(agg["macro_f1_std"], 4),
        "bacc_mean": round(agg["bacc_mean"], 4),
        "mcc_mean": round(agg["mcc_mean"], 4),
        "accuracy_mean": round(agg["accuracy_mean"], 4),
        "n_observations": agg["n_observations"],
    })
df_hdmc_summary = pd.DataFrame(rows)
print(df_hdmc_summary.to_string(index=False))

out_path = os.path.join(TAB_DIR, "hdmc_ringkasan_4_dataset.csv")
df_hdmc_summary.to_csv(out_path, index=False)
print(f"\nTersimpan: {out_path}")
print("\nCatatan: angka ini protokol final 10-fold/1-repeat Part 3, BUKAN 15 kali "
      "pengujian (5-fold/3-repeat) yang sudah dilaporkan ke BIMA di laporan kemajuan. "
      "Angka final untuk naskah jurnal memakai protokol ini.")